# 🔬 BioHub Cell Tracking — 3D Data Exploration & Visualization

**Competition:** [Biohub — Cell Tracking During Development](https://www.kaggle.com/competitions/biohub-cell-tracking-during-development) ($60K Prize)

---

### 📋 What This Notebook Covers

| Section | Description |
|---------|-------------|
| 1 | **Dataset Structure** — Zarr v3 format, dimensions, metadata |
| 2 | **3D Volume Visualization** — Z-projections, slice montages, intensity histograms |
| 3 | **Temporal Analysis** — Frame-to-frame changes, cell count dynamics |
| 4 | **Spatial Statistics** — Voxel intensity distribution, SNR estimation |
| 5 | **Sample Comparison** — Across multiple test volumes |

> 🧬 **Context:** This competition asks us to track individual cells in 3D light-sheet microscopy of developing zebrafish embryos. Understanding the data properties (anisotropy, SNR, cell density) is critical for choosing the right segmentation and tracking approach.

---

In [ ]:
# ── Cell 1: Imports & Setup ──────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import Normalize
import seaborn as sns
import pathlib, warnings, sys

try:
    import zarr
except ImportError:
    !pip install -q zarr
    import zarr

warnings.filterwarnings('ignore')

# ── Dark theme aesthetics ─────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0e1117',
    'axes.facecolor': '#0e1117',
    'axes.edgecolor': '#333333',
    'axes.labelcolor': '#e0e0e0',
    'text.color': '#e0e0e0',
    'xtick.color': '#aaaaaa',
    'ytick.color': '#aaaaaa',
    'grid.color': '#222222',
    'font.size': 11,
    'font.family': 'sans-serif',
})
CMAP_CELLS = 'inferno'
PALETTE = ['#00d2ff', '#ff6b6b', '#51cf66', '#ffd43b', '#cc5de8', '#ff922b']

# Voxel spacing in µm (Z, Y, X) — from competition description
VOXEL_SIZE_UM = np.array([1.625, 0.40625, 0.40625])
ANISOTROPY = VOXEL_SIZE_UM[0] / VOXEL_SIZE_UM[1]  # ~4x

print(f'Voxel size (µm): Z={VOXEL_SIZE_UM[0]}, Y={VOXEL_SIZE_UM[1]}, X={VOXEL_SIZE_UM[2]}')
print(f'Anisotropy ratio: {ANISOTROPY:.1f}x')
print('✅ Setup complete')

In [ ]:
# ── Cell 2: Locate Data ──────────────────────────────────────────────────────────
ON_KAGGLE = pathlib.Path('/kaggle').exists()

if ON_KAGGLE:
    INPUT_BASE = pathlib.Path('/kaggle/input')
    # Search for test directories
    test_dirs = list(INPUT_BASE.rglob('test'))
    if test_dirs:
        DATA_DIR = test_dirs[0]
    else:
        # Fallback: find any .zarr
        zarr_dirs = [p.parent for p in INPUT_BASE.rglob('*.zarr')]
        DATA_DIR = zarr_dirs[0] if zarr_dirs else INPUT_BASE
else:
    DATA_DIR = pathlib.Path('../data/test')

# Find all zarr samples
samples = sorted([p for p in DATA_DIR.iterdir() if p.name.endswith('.zarr') or p.is_dir()]) if DATA_DIR.exists() else []

print(f'Environment: {"Kaggle" if ON_KAGGLE else "Local"}')
print(f'Data directory: {DATA_DIR}')
print(f'Found {len(samples)} samples:')
for s in samples[:10]:
    print(f'  📁 {s.name}')

In [ ]:
# ── Cell 3: Load and Inspect Volume ──────────────────────────────────────────────
def load_volume(path):
    """Open Zarr v3 volume, return 4D array (T, Z, Y, X)."""
    root = zarr.open(str(path), mode='r')
    arr = root['0'] if '0' in root else root
    return arr

if samples:
    sample_path = samples[0]
    vol = load_volume(sample_path)
    
    print(f'\n📊 Sample: {sample_path.name}')
    print(f'   Shape:    {vol.shape}  (T, Z, Y, X)')
    print(f'   Dtype:    {vol.dtype}')
    print(f'   Chunks:   {vol.chunks}')
    print(f'   Frames:   {vol.shape[0]}')
    print(f'   Z-slices: {vol.shape[1]}')
    print(f'   Y-pixels: {vol.shape[2]}')
    print(f'   X-pixels: {vol.shape[3]}')
    
    # Physical dimensions
    phys = np.array(vol.shape[1:]) * VOXEL_SIZE_UM
    print(f'\n   Physical volume per frame:')
    print(f'     Z: {phys[0]:.1f} µm')
    print(f'     Y: {phys[1]:.1f} µm')
    print(f'     X: {phys[2]:.1f} µm')
    print(f'     Total: {np.prod(phys):.0f} µm³')
else:
    print('⚠️ No samples found — please attach the competition dataset.')

In [ ]:
# ── Cell 4: 3D Volume Visualization — Z-Projections ──────────────────────────────
if samples:
    frame0 = np.asarray(vol[0])  # First frame
    
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    
    # Max intensity projection along Z
    mip_z = frame0.max(axis=0)
    axes[0].imshow(mip_z, cmap=CMAP_CELLS, aspect='equal')
    axes[0].set_title('Max Intensity Projection (Z-axis)', fontsize=13, fontweight='bold')
    axes[0].set_xlabel('X (pixels)')
    axes[0].set_ylabel('Y (pixels)')
    
    # Max intensity projection along Y
    mip_y = frame0.max(axis=1)
    axes[1].imshow(mip_y, cmap=CMAP_CELLS, aspect=ANISOTROPY)
    axes[1].set_title('Max Intensity Projection (Y-axis)', fontsize=13, fontweight='bold')
    axes[1].set_xlabel('X (pixels)')
    axes[1].set_ylabel('Z (pixels)')
    
    # Mean intensity projection along Z
    mean_z = frame0.mean(axis=0)
    axes[2].imshow(mean_z, cmap=CMAP_CELLS, aspect='equal')
    axes[2].set_title('Mean Intensity Projection (Z-axis)', fontsize=13, fontweight='bold')
    axes[2].set_xlabel('X (pixels)')
    axes[2].set_ylabel('Y (pixels)')
    
    plt.suptitle(f'📸 Volume Projections — {sample_path.name} (Frame 0)', 
                fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()
    
    print(f'Intensity range: [{frame0.min()}, {frame0.max()}]')
    print(f'Mean intensity:  {frame0.mean():.2f}')
    print(f'Std intensity:   {frame0.std():.2f}')

In [ ]:
# ── Cell 5: Z-Slice Montage ──────────────────────────────────────────────────────
if samples:
    n_z = frame0.shape[0]
    n_show = min(16, n_z)
    indices = np.linspace(0, n_z - 1, n_show, dtype=int)
    
    cols = 4
    rows = (n_show + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(20, 5 * rows))
    axes = axes.flatten()
    
    vmin, vmax = np.percentile(frame0, [1, 99])
    
    for idx, z in enumerate(indices):
        axes[idx].imshow(frame0[z], cmap=CMAP_CELLS, vmin=vmin, vmax=vmax)
        axes[idx].set_title(f'Z = {z} ({z * VOXEL_SIZE_UM[0]:.1f} µm)', 
                           fontsize=10, fontweight='bold')
        axes[idx].axis('off')
    
    for idx in range(len(indices), len(axes)):
        axes[idx].set_visible(False)
    
    plt.suptitle(f'🔍 Z-Slice Montage — {sample_path.name} (Frame 0, {n_z} total slices)', 
                fontsize=16, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Cell 6: Intensity Distribution & SNR Analysis ─────────────────────────────────
if samples:
    fig, axes = plt.subplots(1, 3, figsize=(20, 5))
    
    # Full histogram
    flat = frame0.ravel()
    axes[0].hist(flat, bins=200, color=PALETTE[0], alpha=0.7, density=True, log=True)
    axes[0].set_title('Voxel Intensity Distribution (log scale)', fontsize=13, fontweight='bold')
    axes[0].set_xlabel('Intensity')
    axes[0].set_ylabel('Density')
    
    # Per-Z-slice mean intensity
    z_means = [frame0[z].mean() for z in range(n_z)]
    z_stds = [frame0[z].std() for z in range(n_z)]
    z_positions = np.arange(n_z) * VOXEL_SIZE_UM[0]
    
    axes[1].fill_between(z_positions, 
                         np.array(z_means) - np.array(z_stds), 
                         np.array(z_means) + np.array(z_stds), 
                         alpha=0.2, color=PALETTE[2])
    axes[1].plot(z_positions, z_means, color=PALETTE[2], linewidth=2)
    axes[1].set_title('Mean Intensity per Z-slice', fontsize=13, fontweight='bold')
    axes[1].set_xlabel('Z position (µm)')
    axes[1].set_ylabel('Mean intensity')
    
    # Signal-to-noise estimation
    bg_threshold = np.percentile(flat, 50)  # Below median = background
    bg = flat[flat <= bg_threshold]
    fg = flat[flat > bg_threshold]
    snr = fg.mean() / (bg.std() + 1e-8)
    
    axes[2].hist(bg, bins=100, alpha=0.6, color=PALETTE[0], label=f'Background (n={len(bg):,})', density=True)
    axes[2].hist(fg, bins=100, alpha=0.6, color=PALETTE[1], label=f'Signal (n={len(fg):,})', density=True)
    axes[2].axvline(bg_threshold, color='white', linestyle='--', linewidth=1.5, label=f'Threshold={bg_threshold:.0f}')
    axes[2].set_title(f'Background vs Signal (SNR ≈ {snr:.1f})', fontsize=13, fontweight='bold')
    axes[2].set_xlabel('Intensity')
    axes[2].legend(fontsize=9)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Cell 7: Temporal Dynamics ────────────────────────────────────────────────────
if samples:
    n_frames = min(vol.shape[0], 20)  # Analyze up to 20 frames
    frame_stats = []
    
    for t in range(n_frames):
        frame = np.asarray(vol[t])
        frame_stats.append({
            'frame': t,
            'mean': frame.mean(),
            'std': frame.std(),
            'max': frame.max(),
            'p99': np.percentile(frame, 99),
            'nonzero_frac': (frame > 0).mean(),
        })
    
    import pandas as pd
    stats_df = pd.DataFrame(frame_stats)
    
    fig, axes = plt.subplots(1, 3, figsize=(20, 5))
    
    axes[0].plot(stats_df['frame'], stats_df['mean'], '-o', color=PALETTE[0], 
                linewidth=2, markersize=5, label='Mean')
    axes[0].fill_between(stats_df['frame'], 
                         stats_df['mean'] - stats_df['std'],
                         stats_df['mean'] + stats_df['std'],
                         alpha=0.15, color=PALETTE[0])
    axes[0].set_title('Mean Intensity Over Time', fontsize=13, fontweight='bold')
    axes[0].set_xlabel('Frame')
    axes[0].set_ylabel('Intensity')
    
    axes[1].plot(stats_df['frame'], stats_df['p99'], '-s', color=PALETTE[1], 
                linewidth=2, markersize=5)
    axes[1].set_title('99th Percentile Intensity Over Time', fontsize=13, fontweight='bold')
    axes[1].set_xlabel('Frame')
    axes[1].set_ylabel('Intensity (P99)')
    
    axes[2].bar(stats_df['frame'], stats_df['nonzero_frac'] * 100, 
               color=PALETTE[2], edgecolor='white', linewidth=0.3)
    axes[2].set_title('Non-zero Voxel Fraction Over Time', fontsize=13, fontweight='bold')
    axes[2].set_xlabel('Frame')
    axes[2].set_ylabel('Non-zero (%)')
    
    plt.suptitle(f'⏱️ Temporal Dynamics — {sample_path.name} ({n_frames} frames)',
                fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Cell 8: Multi-Sample Comparison ──────────────────────────────────────────────
if len(samples) > 1:
    sample_info = []
    for sp in samples[:8]:  # Compare up to 8 samples
        try:
            v = load_volume(sp)
            f0 = np.asarray(v[0])
            sample_info.append({
                'name': sp.name,
                'frames': v.shape[0],
                'z_slices': v.shape[1],
                'height': v.shape[2],
                'width': v.shape[3],
                'mean_int': f0.mean(),
                'std_int': f0.std(),
                'max_int': f0.max(),
            })
        except Exception as e:
            print(f'  ⚠️ Error loading {sp.name}: {e}')
    
    if sample_info:
        info_df = pd.DataFrame(sample_info)
        
        fig, axes = plt.subplots(1, 3, figsize=(20, 5))
        
        x = range(len(info_df))
        names = [n[:12] for n in info_df['name']]
        
        axes[0].bar(x, info_df['frames'], color=PALETTE[0], edgecolor='white', linewidth=0.3)
        axes[0].set_xticks(x)
        axes[0].set_xticklabels(names, rotation=45, ha='right', fontsize=9)
        axes[0].set_title('Frames per Sample', fontsize=13, fontweight='bold')
        
        axes[1].bar(x, info_df['z_slices'], color=PALETTE[2], edgecolor='white', linewidth=0.3)
        axes[1].set_xticks(x)
        axes[1].set_xticklabels(names, rotation=45, ha='right', fontsize=9)
        axes[1].set_title('Z-Slices per Sample', fontsize=13, fontweight='bold')
        
        axes[2].bar(x, info_df['mean_int'], color=PALETTE[3], edgecolor='white', linewidth=0.3)
        axes[2].set_xticks(x)
        axes[2].set_xticklabels(names, rotation=45, ha='right', fontsize=9)
        axes[2].set_title('Mean Intensity (Frame 0)', fontsize=13, fontweight='bold')
        
        plt.suptitle('📊 Cross-Sample Comparison', fontsize=16, fontweight='bold', y=1.02)
        plt.tight_layout()
        plt.show()
        
        print('\n📋 Sample Details:')
        display(info_df)
else:
    print('Only 1 sample available — skipping cross-sample comparison.')

---

## 📌 Key Takeaways

| Property | Observation |
|----------|-------------|
| **Format** | Zarr v3, 4D arrays (T, Z, Y, X) |
| **Anisotropy** | ~4x (Z resolution ≈ 1.625 µm, XY ≈ 0.406 µm) |
| **Implications** | Must account for anisotropy in 3D segmentation (e.g., Cellpose `anisotropy` parameter) |
| **SNR** | Varies by sample — blob fallback may be needed for low-SNR volumes |
| **Temporal stability** | Intensity is relatively stable across frames |

### 🔮 Recommendations for Tracking Pipeline
1. **Use Cellpose `cyto3` with `do_3D=True`** and set `anisotropy=4.0`
2. **Post-process** with size filtering (50–50K voxels) to remove noise
3. **Hungarian matching** with physical-distance cutoff (10 µm) for linking
4. **Gap-2 bridging** to handle temporary disappearances

---

**If you found this notebook helpful, please upvote! 👍**